# **TensorFlow Bencmark**

## Setup & GPU Info

In [1]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 1: Setup & GPU Info
# ============================================================
import tensorflow as tf
import numpy as np
import time
import json
import gc
import subprocess
import datetime

print("=" * 70)
print("TENSORFLOW BENCHMARK SUITE (ROCm)")
print("=" * 70)

print(f"\nTensorFlow version: {tf.__version__}")
print(f"Built with CUDA/ROCm: {tf.test.is_built_with_cuda()}")

# GPU detection
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)}")
for i, gpu in enumerate(gpus):
    print(f"  GPU {i}: {gpu}")

if gpus:
    try:
        # Allow memory growth
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Get device details
        details = tf.config.experimental.get_device_details(gpus[0])
        print(f"Device details: {details}")
    except Exception as e:
        print(f"GPU config note: {e}")

# Try ROCm info
try:
    result = subprocess.run(['rocm-smi', '--showid', '--showtemp', '--showuse'],
                          capture_output=True, text=True, timeout=10)
    print(f"\nROCm SMI Output:\n{result.stdout}")
except:
    print("\nCould not run rocm-smi")

# Results storage
results = {
    'framework': 'TensorFlow',
    'version': tf.__version__,
    'device': str(gpus[0]) if gpus else 'CPU',
    'benchmarks': {}
}

# Utility functions
def benchmark_fn(fn, num_runs=100, warmup_runs=20, label=""):
    """Benchmark a function"""
    # Warmup
    for _ in range(warmup_runs):
        fn()

    # Ensure GPU sync via explicit sync
    if gpus:
        # Force synchronization
        dummy = tf.constant(1.0)
        _ = dummy.numpy()

    times = []
    for _ in range(num_runs):
        if gpus:
            _ = tf.constant(1.0).numpy()  # sync before start
        start = time.perf_counter()
        result = fn()
        # Force sync by reading result
        if isinstance(result, tf.Tensor):
            _ = result.numpy() if result.shape == () else result[0, 0].numpy() if len(result.shape) >= 2 else result[0].numpy()
        elif isinstance(result, (list, tuple)):
            _ = result[0] if len(result) > 0 else None
        end = time.perf_counter()
        times.append(end - start)

    times = np.array(times)
    r = {
        'mean_ms': float(np.mean(times) * 1000),
        'std_ms': float(np.std(times) * 1000),
        'min_ms': float(np.min(times) * 1000),
        'max_ms': float(np.max(times) * 1000),
        'median_ms': float(np.median(times) * 1000),
        'p95_ms': float(np.percentile(times, 95) * 1000),
        'p99_ms': float(np.percentile(times, 99) * 1000),
        'num_runs': num_runs
    }
    if label:
        print(f"  {label}: {r['mean_ms']:.3f} ± {r['std_ms']:.3f} ms "
              f"(min={r['min_ms']:.3f}, median={r['median_ms']:.3f})")
    return r

# More accurate benchmark that uses tf.timestamp for GPU-side timing
def benchmark_fn_gpu(fn, num_runs=100, warmup_runs=20, label=""):
    """Benchmark using eager mode with proper synchronization"""
    for _ in range(warmup_runs):
        r = fn()
        if isinstance(r, tf.Tensor):
            r.numpy()

    times = []
    for _ in range(num_runs):
        # Force GPU sync
        tf.ones([]).numpy()
        start = time.perf_counter()
        r = fn()
        if isinstance(r, tf.Tensor):
            r.numpy()
        end = time.perf_counter()
        times.append(end - start)

    times = np.array(times)
    result = {
        'mean_ms': float(np.mean(times) * 1000),
        'std_ms': float(np.std(times) * 1000),
        'min_ms': float(np.min(times) * 1000),
        'max_ms': float(np.max(times) * 1000),
        'median_ms': float(np.median(times) * 1000),
        'p95_ms': float(np.percentile(times, 95) * 1000),
        'p99_ms': float(np.percentile(times, 99) * 1000),
        'num_runs': num_runs
    }
    if label:
        print(f"  {label}: {result['mean_ms']:.3f} ± {result['std_ms']:.3f} ms "
              f"(min={result['min_ms']:.3f}, median={result['median_ms']:.3f})")
    return result

def warmup_gpu():
    if gpus:
        for _ in range(10):
            a = tf.random.normal([256, 256])
            b = tf.random.normal([256, 256])
            c = tf.matmul(a, b)
            c.numpy()
        del a, b, c

print("\n✅ Setup complete!")

2026-02-17 11:33:49.944498: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TENSORFLOW BENCHMARK SUITE (ROCm)

TensorFlow version: 2.20.0-dev0+selfbuilt
Built with CUDA/ROCm: False
GPUs detected: 1
  GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Device details: {'device_name': 'AMD Radeon RX 6800S'}

ROCm SMI Output:


============================ ROCm System Management Interface ============================
=========================================== ID ===========================================
GPU[0]		: Device Name: 		AMD Radeon RX 6800S
GPU[0]		: Device ID: 		0x73ef
GPU[0]		: Device Rev: 		0xc0
GPU[0]		: Subsystem ID: 	0x1dcc
GPU[0]		: GUID: 		45168
GPU[1]		: Device Name: 		AMD Radeon 680M
GPU[1]		: Device ID: 		0x1681
GPU[1]		: Device Rev: 		0xc7
GPU[1]		: Subsystem ID: 	0x1dcc
GPU[1]		: GUID: 		56463
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 61.0
GPU[0]		: Temperature (Sensor junction) (C): 61.0
GPU[0]		: Temperature (Sensor memory) (C): 5

I0000 00:00:1771302834.607633  130667 gpu_device.cc:2411] Ignoring visible gpu device (device: 1, name: AMD Radeon 680M, pci bus id: 0000:07:00.0) with core count: 6. The minimum required count is 8. You can adjust this requirement with the env var TF_MIN_GPU_MULTIPROCESSOR_COUNT.


In [2]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 2: Matrix Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 1: MATRIX OPERATIONS")
print("=" * 70)

results['benchmarks']['matmul'] = {}

sizes = [256, 512, 1024, 2048, 4096, 8192]
dtypes_config = {
    'float32': tf.float32,
    'float16': tf.float16,
}

# Check bfloat16
try:
    test = tf.random.normal([2, 2], dtype=tf.bfloat16)
    dtypes_config['bfloat16'] = tf.bfloat16
    del test
except:
    print("bfloat16 not supported")

for dtype_name, dtype in dtypes_config.items():
    print(f"\n--- Matrix Multiplication ({dtype_name}) ---")
    results['benchmarks']['matmul'][dtype_name] = {}

    for size in sizes:
        try:
            warmup_gpu()
            a = tf.random.normal([size, size], dtype=dtype if dtype != tf.float16 else tf.float32)
            b = tf.random.normal([size, size], dtype=dtype if dtype != tf.float16 else tf.float32)
            if dtype == tf.float16:
                a = tf.cast(a, tf.float16)
                b = tf.cast(b, tf.float16)

            @tf.function
            def matmul_fn():
                return tf.matmul(a, b)

            num_runs = 200 if size <= 1024 else (100 if size <= 4096 else 50)
            r = benchmark_fn_gpu(matmul_fn, num_runs=num_runs, warmup_runs=20,
                               label=f"  Size {size}x{size}")

            flops = 2 * size * size * size
            tflops = flops / (r['mean_ms'] / 1000) / 1e12
            r['tflops'] = float(tflops)
            print(f"    → {tflops:.2f} TFLOPS")

            results['benchmarks']['matmul'][dtype_name][str(size)] = r

            del a, b
            gc.collect()
        except Exception as e:
            print(f"  Size {size}x{size}: SKIPPED ({e})")
            results['benchmarks']['matmul'][dtype_name][str(size)] = {'error': str(e)}

# Batched matmul
print(f"\n--- Batched Matrix Multiplication (float32) ---")
results['benchmarks']['batched_matmul'] = {}
batch_configs = [(32, 512, 512), (64, 256, 256), (128, 128, 128), (16, 1024, 1024)]

for batch, m, n in batch_configs:
    try:
        warmup_gpu()
        a = tf.random.normal([batch, m, n])
        b = tf.random.normal([batch, n, m])

        @tf.function
        def batched_matmul_fn():
            return tf.matmul(a, b)

        r = benchmark_fn_gpu(batched_matmul_fn, num_runs=100, warmup_runs=20,
                           label=f"  Batch={batch}, Size={m}x{n}")

        flops = 2 * batch * m * n * m
        tflops = flops / (r['mean_ms'] / 1000) / 1e12
        r['tflops'] = float(tflops)
        print(f"    → {tflops:.2f} TFLOPS")

        key = f"b{batch}_{m}x{n}"
        results['benchmarks']['batched_matmul'][key] = r

        del a, b
        gc.collect()
    except Exception as e:
        print(f"  Config ({batch},{m},{n}): SKIPPED ({e})")

gc.collect()
print("\n✅ Matrix operations benchmark complete!")

BENCHMARK 1: MATRIX OPERATIONS


I0000 00:00:1771302834.924814  130667 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6904 MB memory:  -> device: 0, name: AMD Radeon RX 6800S, pci bus id: 0000:03:00.0



--- Matrix Multiplication (float32) ---
    Size 256x256: 0.388 ± 0.117 ms (min=0.256, median=0.349)
    → 0.09 TFLOPS
    Size 512x512: 0.966 ± 0.375 ms (min=0.441, median=0.934)
    → 0.28 TFLOPS
    Size 1024x1024: 1.542 ± 0.309 ms (min=1.092, median=1.480)
    → 1.39 TFLOPS
    Size 2048x2048: 7.276 ± 1.545 ms (min=6.069, median=6.904)
    → 2.36 TFLOPS


2026-02-17 11:33:57.830323: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67108864 exceeds 10% of free system memory.
2026-02-17 11:33:57.930086: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67108864 exceeds 10% of free system memory.
2026-02-17 11:33:57.996522: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67108864 exceeds 10% of free system memory.
2026-02-17 11:33:58.097767: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67108864 exceeds 10% of free system memory.
2026-02-17 11:33:58.170939: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67108864 exceeds 10% of free system memory.


    Size 4096x4096: 61.282 ± 4.508 ms (min=54.512, median=60.722)
    → 2.24 TFLOPS
    Size 8192x8192: 291.733 ± 10.054 ms (min=278.281, median=288.788)
    → 3.77 TFLOPS

--- Matrix Multiplication (float16) ---
    Size 256x256: 0.328 ± 0.062 ms (min=0.237, median=0.316)
    → 0.10 TFLOPS
    Size 512x512: 0.410 ± 0.162 ms (min=0.288, median=0.378)
    → 0.66 TFLOPS
    Size 1024x1024: 0.733 ± 0.145 ms (min=0.554, median=0.679)
    → 2.93 TFLOPS
    Size 2048x2048: 3.073 ± 0.692 ms (min=2.473, median=2.848)
    → 5.59 TFLOPS
    Size 4096x4096: 30.299 ± 3.060 ms (min=26.000, median=29.105)
    → 4.54 TFLOPS
    Size 8192x8192: 145.170 ± 6.772 ms (min=134.618, median=144.718)
    → 7.57 TFLOPS

--- Matrix Multiplication (bfloat16) ---
    Size 256x256: 0.353 ± 0.072 ms (min=0.251, median=0.335)
    → 0.09 TFLOPS
    Size 512x512: 0.441 ± 0.088 ms (min=0.333, median=0.418)
    → 0.61 TFLOPS
    Size 1024x1024: 1.061 ± 0.139 ms (min=0.897, median=1.012)
    → 2.02 TFLOPS
    Size 2048x2

In [3]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 3: Element-wise & Reduction Ops
# ============================================================
print("=" * 70)
print("BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS")
print("=" * 70)

results['benchmarks']['elementwise'] = {}
results['benchmarks']['reductions'] = {}

tensor_sizes = [
    (1_000_000,),
    (10_000_000,),
    (50_000_000,),
    (100_000_000,),
]

print("\n--- Element-wise Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['elementwise'][label] = {}

    warmup_gpu()
    a = tf.random.normal(shape)
    b = tf.random.normal(shape)

    ops = {
        'add': lambda: tf.add(a, b),
        'mul': lambda: tf.multiply(a, b),
        'exp': lambda: tf.exp(a),
        'sin': lambda: tf.sin(a),
        'sigmoid': lambda: tf.sigmoid(a),
        'tanh': lambda: tf.tanh(a),
        'relu': lambda: tf.nn.relu(a),
        'gelu': lambda: tf.nn.gelu(a),
        'silu': lambda: tf.nn.silu(a),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            tf_fn = tf.function(op_fn)
            r = benchmark_fn_gpu(tf_fn, num_runs=100, warmup_runs=20,
                               label=f"    {op_name}")
            bytes_moved = numel * 4 * 2
            if op_name in ['add', 'mul']:
                bytes_moved = numel * 4 * 3
            bandwidth_gbs = bytes_moved / (r['mean_ms'] / 1000) / 1e9
            r['bandwidth_gbs'] = float(bandwidth_gbs)
            results['benchmarks']['elementwise'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a, b
    gc.collect()

# Reduction operations
print(f"\n--- Reduction Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['reductions'][label] = {}

    warmup_gpu()
    a = tf.random.normal(shape)

    ops = {
        'sum': lambda: tf.reduce_sum(a),
        'mean': lambda: tf.reduce_mean(a),
        'max': lambda: tf.reduce_max(a),
        'min': lambda: tf.reduce_min(a),
        'std': lambda: tf.math.reduce_std(a),
        'norm': lambda: tf.norm(a),
        'argmax': lambda: tf.argmax(a),
        'sort': lambda: tf.sort(a),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            tf_fn = tf.function(op_fn)
            num_runs = 50 if op_name == 'sort' else 100
            r = benchmark_fn_gpu(tf_fn, num_runs=num_runs, warmup_runs=10,
                               label=f"    {op_name}")
            results['benchmarks']['reductions'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a
    gc.collect()

print("\n✅ Element-wise & reduction benchmark complete!")

BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS

--- Element-wise Operations ---

  Tensor size: 1M elements
      add: 0.959 ± 0.274 ms (min=0.683, median=0.873)
      mul: 1.013 ± 0.322 ms (min=0.637, median=0.931)
      exp: 0.926 ± 0.239 ms (min=0.640, median=0.827)
      sin: 0.898 ± 0.232 ms (min=0.631, median=0.783)
      sigmoid: 0.946 ± 0.247 ms (min=0.700, median=0.824)
      tanh: 0.946 ± 0.274 ms (min=0.636, median=0.809)
      relu: 0.959 ± 0.244 ms (min=0.680, median=0.879)
      gelu: 1.050 ± 0.329 ms (min=0.714, median=0.892)
      silu: 0.951 ± 0.298 ms (min=0.643, median=0.818)

  Tensor size: 10M elements
      add: 24.206 ± 0.996 ms (min=21.985, median=24.239)
      mul: 24.129 ± 1.475 ms (min=20.884, median=24.118)
      exp: 24.620 ± 1.456 ms (min=19.946, median=24.487)
      sin: 25.536 ± 1.144 ms (min=22.757, median=25.552)
      sigmoid: 24.215 ± 1.506 ms (min=20.519, median=23.973)
      tanh: 25.312 ± 1.485 ms (min=22.330, median=25.410)
      relu: 25.656 ±

In [4]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 4: Convolution Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 3: CONVOLUTION OPERATIONS")
print("=" * 70)

results['benchmarks']['conv2d'] = {}

conv_configs = [
    (32, 3, 64, 224, 224, 7, 2, 'same', "ResNet-stem"),
    (32, 64, 64, 56, 56, 3, 1, 'same', "ResNet-block1"),
    (32, 128, 128, 28, 28, 3, 1, 'same', "ResNet-block2"),
    (32, 256, 256, 14, 14, 3, 1, 'same', "ResNet-block3"),
    (32, 512, 512, 7, 7, 3, 1, 'same', "ResNet-block4"),
    (64, 3, 32, 32, 32, 3, 1, 'same', "CIFAR-style"),
    (16, 3, 64, 224, 224, 3, 1, 'same', "VGG-style"),
    (1, 64, 64, 512, 512, 3, 1, 'same', "HighRes-single"),
]

for batch, in_ch, out_ch, H, W, kernel, stride, padding, label in conv_configs:
    try:
        warmup_gpu()

        # NHWC format (TF default)
        x = tf.random.normal([batch, H, W, in_ch])
        conv_layer = tf.keras.layers.Conv2D(
            out_ch, kernel, strides=stride, padding=padding, use_bias=False
        )
        # Build the layer
        _ = conv_layer(x)

        @tf.function
        def conv_forward():
            return conv_layer(x)

        print(f"\n  {label}: input=({batch},{in_ch},{H},{W}), "
              f"kernel={kernel}x{kernel}, out_ch={out_ch}")

        r_fwd = benchmark_fn_gpu(conv_forward, num_runs=100, warmup_runs=20,
                                label=f"    Forward")

        # Forward + Backward
        @tf.function
        def conv_fwd_bwd():
            with tf.GradientTape() as tape:
                out = conv_layer(x)
                loss = tf.reduce_sum(out)
            grads = tape.gradient(loss, conv_layer.trainable_variables)
            return loss, grads

        r_fwd_bwd = benchmark_fn_gpu(conv_fwd_bwd, num_runs=50, warmup_runs=10,
                                    label=f"    Fwd+Bwd")

        results['benchmarks']['conv2d'][label] = {
            'forward': r_fwd,
            'fwd_bwd': r_fwd_bwd,
            'config': {
                'batch': batch, 'in_ch': in_ch, 'out_ch': out_ch,
                'H': H, 'W': W, 'kernel': kernel, 'stride': stride
            }
        }

        del x, conv_layer
        gc.collect()

    except Exception as e:
        print(f"  {label}: SKIPPED ({e})")
        results['benchmarks']['conv2d'][label] = {'error': str(e)}

gc.collect()
print("\n✅ Convolution benchmark complete!")

BENCHMARK 3: CONVOLUTION OPERATIONS

  ResNet-stem: input=(32,3,224,224), kernel=7x7, out_ch=64
      Forward: 59.228 ± 2.073 ms (min=55.076, median=58.899)
      Fwd+Bwd: 1.315 ± 0.182 ms (min=0.976, median=1.326)

  ResNet-block1: input=(32,64,56,56), kernel=3x3, out_ch=64
      Forward: 7.266 ± 0.640 ms (min=6.301, median=7.212)
      Fwd+Bwd: 0.471 ± 0.161 ms (min=0.289, median=0.424)

  ResNet-block2: input=(32,128,28,28), kernel=3x3, out_ch=128
      Forward: 2.974 ± 0.285 ms (min=2.502, median=2.940)
      Fwd+Bwd: 0.485 ± 0.543 ms (min=0.261, median=0.359)

  ResNet-block3: input=(32,256,14,14), kernel=3x3, out_ch=256
      Forward: 2.246 ± 0.374 ms (min=1.764, median=2.141)
      Fwd+Bwd: 0.294 ± 0.061 ms (min=0.232, median=0.270)

  ResNet-block4: input=(32,512,7,7), kernel=3x3, out_ch=512
      Forward: 3.024 ± 0.263 ms (min=2.489, median=3.035)
      Fwd+Bwd: 0.363 ± 0.138 ms (min=0.252, median=0.308)

  CIFAR-style: input=(64,3,32,32), kernel=3x3, out_ch=32
      Forward: 

In [5]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 5: Common Layer Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 4: COMMON LAYER OPERATIONS")
print("=" * 70)

results['benchmarks']['layers'] = {}

warmup_gpu()

# --- Linear (Dense) Layer ---
print("\n--- Dense (Linear) Layers ---")
linear_configs = [
    (128, 768, 768, "BERT-hidden"),
    (128, 768, 3072, "BERT-FFN-up"),
    (128, 3072, 768, "BERT-FFN-down"),
    (32, 1024, 4096, "Large-FFN-up"),
    (32, 4096, 1024, "Large-FFN-down"),
    (1, 4096, 4096, "LLM-single"),
]

results['benchmarks']['layers']['linear'] = {}
for batch, in_feat, out_feat, label in linear_configs:
    try:
        warmup_gpu()
        x = tf.random.normal([batch, in_feat])
        layer = tf.keras.layers.Dense(out_feat, use_bias=True)
        _ = layer(x)  # build

        @tf.function
        def linear_fwd():
            return layer(x)

        @tf.function
        def linear_fwd_bwd():
            with tf.GradientTape() as tape:
                out = layer(x)
                loss = tf.reduce_sum(out)
            grads = tape.gradient(loss, layer.trainable_variables)
            return loss, grads

        r_fwd = benchmark_fn_gpu(linear_fwd, num_runs=200, warmup_runs=30,
                                label=f"  {label} fwd")
        r_bwd = benchmark_fn_gpu(linear_fwd_bwd, num_runs=100, warmup_runs=20,
                                label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['linear'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del x, layer
        gc.collect()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- BatchNorm ---
print("\n--- BatchNorm ---")
results['benchmarks']['layers']['batchnorm'] = {}
bn_configs = [
    (32, 64, 56, 56, "BN-early"),
    (32, 256, 14, 14, "BN-mid"),
    (32, 512, 7, 7, "BN-late"),
]

for batch, ch, H, W, label in bn_configs:
    try:
        warmup_gpu()
        x = tf.random.normal([batch, H, W, ch])  # NHWC
        bn = tf.keras.layers.BatchNormalization()
        _ = bn(x, training=True)

        @tf.function
        def bn_fwd():
            return bn(x, training=True)

        r = benchmark_fn_gpu(bn_fwd, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['batchnorm'][label] = r
        del x, bn
        gc.collect()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- LayerNorm ---
print("\n--- LayerNorm ---")
results['benchmarks']['layers']['layernorm'] = {}
ln_configs = [
    ((128, 128, 768), "BERT-LN"),
    ((32, 256, 1024), "Large-LN"),
    ((1, 2048, 4096), "LLM-LN"),
]

for shape, label in ln_configs:
    try:
        warmup_gpu()
        x = tf.random.normal(shape)
        ln = tf.keras.layers.LayerNormalization()
        _ = ln(x)

        @tf.function
        def ln_fwd():
            return ln(x)

        r = benchmark_fn_gpu(ln_fwd, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['layernorm'][label] = r
        del x, ln
        gc.collect()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Multi-Head Attention ---
print("\n--- Multi-Head Attention ---")
results['benchmarks']['layers']['attention'] = {}
attn_configs = [
    (32, 128, 768, 12, "BERT-attn"),
    (16, 256, 768, 12, "BERT-long-attn"),
    (8, 512, 1024, 16, "Large-attn"),
    (1, 1024, 1024, 16, "LLM-attn"),
    (1, 2048, 768, 12, "Long-seq-attn"),
]

for batch, seq_len, embed, heads, label in attn_configs:
    try:
        warmup_gpu()
        mha = tf.keras.layers.MultiHeadAttention(
            num_heads=heads, key_dim=embed // heads
        )
        x = tf.random.normal([batch, seq_len, embed])
        _ = mha(x, x)  # build

        @tf.function
        def attn_fwd():
            return mha(x, x)

        @tf.function
        def attn_fwd_bwd():
            with tf.GradientTape() as tape:
                out = mha(x, x)
                loss = tf.reduce_sum(out)
            grads = tape.gradient(loss, mha.trainable_variables)
            return loss, grads

        r_fwd = benchmark_fn_gpu(attn_fwd, num_runs=50, warmup_runs=10,
                                label=f"  {label} fwd")
        r_bwd = benchmark_fn_gpu(attn_fwd_bwd, num_runs=30, warmup_runs=10,
                                label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['attention'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del mha, x
        gc.collect()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Softmax ---
print("\n--- Softmax ---")
results['benchmarks']['layers']['softmax'] = {}
for size in [(128, 128, 768), (32, 256, 1024), (8, 512, 4096)]:
    try:
        warmup_gpu()
        x = tf.random.normal(size)
        label = f"{'x'.join(map(str, size))}"

        @tf.function
        def softmax_fn():
            return tf.nn.softmax(x, axis=-1)

        r = benchmark_fn_gpu(softmax_fn, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['softmax'][label] = r
        del x
        gc.collect()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

gc.collect()
print("\n✅ Layer operations benchmark complete!")

BENCHMARK 4: COMMON LAYER OPERATIONS

--- Dense (Linear) Layers ---
    BERT-hidden fwd: 0.413 ± 0.112 ms (min=0.299, median=0.374)
    BERT-hidden fwd+bwd: 0.191 ± 0.028 ms (min=0.153, median=0.184)
    BERT-FFN-up fwd: 0.639 ± 0.143 ms (min=0.497, median=0.583)
    BERT-FFN-up fwd+bwd: 0.201 ± 0.033 ms (min=0.163, median=0.190)
    BERT-FFN-down fwd: 0.494 ± 0.056 ms (min=0.405, median=0.481)
    BERT-FFN-down fwd+bwd: 0.203 ± 0.065 ms (min=0.165, median=0.186)
    Large-FFN-up fwd: 0.437 ± 0.072 ms (min=0.329, median=0.415)
    Large-FFN-up fwd+bwd: 0.194 ± 0.028 ms (min=0.162, median=0.185)
    Large-FFN-down fwd: 0.505 ± 0.087 ms (min=0.404, median=0.486)
    Large-FFN-down fwd+bwd: 0.221 ± 0.036 ms (min=0.172, median=0.209)
    LLM-single fwd: 0.561 ± 0.076 ms (min=0.472, median=0.542)
    LLM-single fwd+bwd: 0.244 ± 0.059 ms (min=0.176, median=0.226)

--- BatchNorm ---
    BN-early: 7.073 ± 0.336 ms (min=6.508, median=7.061)
    BN-mid: 1.676 ± 0.304 ms (min=1.245, median=1.627)

In [6]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 6: CNN Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)")
print("=" * 70)

results['benchmarks']['cnn_training'] = {}

def make_resnet_model(num_classes=1000):
    """Create a ResNet-18-like model"""
    inputs = tf.keras.Input(shape=(224, 224, 3))

    # Stem
    x = tf.keras.layers.Conv2D(64, 7, strides=2, padding='same', use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(3, strides=2, padding='same')(x)

    def residual_block(x, filters, stride=1):
        shortcut = x

        y = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
        y = tf.keras.layers.BatchNormalization()(y)
        y = tf.keras.layers.ReLU()(y)
        y = tf.keras.layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(y)
        y = tf.keras.layers.BatchNormalization()(y)

        if stride != 1 or shortcut.shape[-1] != filters:
            shortcut = tf.keras.layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
            shortcut = tf.keras.layers.BatchNormalization()(shortcut)

        y = tf.keras.layers.Add()([y, shortcut])
        y = tf.keras.layers.ReLU()(y)
        return y

    # Layer 1
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    # Layer 2
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)
    # Layer 3
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)
    # Layer 4
    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(num_classes)(x)

    return tf.keras.Model(inputs, outputs)

# Count parameters
test_model = make_resnet_model()
total_params = test_model.count_params()
print(f"\nModel: SimpleResNet-18-like")
print(f"Total parameters: {total_params:,}")
del test_model
gc.collect()

# Training benchmark
batch_sizes = [16, 32, 64]
num_classes = 1000
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for batch_size in batch_sizes:
    label = f"batch_{batch_size}"
    print(f"\n--- Batch Size: {batch_size} ---")

    try:
        warmup_gpu()

        model = make_resnet_model(num_classes)
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

        x = tf.random.normal([batch_size, 224, 224, 3])
        y = tf.random.uniform([batch_size], 0, num_classes, dtype=tf.int32)

        # Inference
        @tf.function
        def inference_fn():
            return model(x, training=False)

        r_infer = benchmark_fn_gpu(inference_fn, num_runs=50, warmup_runs=10,
                                  label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_imgs_per_sec'] = float(throughput_infer)
        print(f"    → {throughput_infer:.1f} images/sec")

        # Training step
        @tf.function
        def train_step():
            with tf.GradientTape() as tape:
                predictions = model(x, training=True)
                loss = loss_fn(y, predictions)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            return loss

        r_train = benchmark_fn_gpu(train_step, num_runs=50, warmup_runs=10,
                                  label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_imgs_per_sec'] = float(throughput_train)
        print(f"    → {throughput_train:.1f} images/sec")

        results['benchmarks']['cnn_training'][label] = {
            'inference': r_infer,
            'training': r_train,
        }

        del model, optimizer, x, y
        gc.collect()

    except Exception as e:
        print(f"  SKIPPED: {e}")
        results['benchmarks']['cnn_training'][label] = {'error': str(e)}

# Mixed Precision
print(f"\n--- Mixed Precision Training ---")
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

    batch_size = 32
    warmup_gpu()

    model = make_resnet_model(num_classes)
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)

    x = tf.random.normal([batch_size, 224, 224, 3])
    y = tf.random.uniform([batch_size], 0, num_classes, dtype=tf.int32)

    @tf.function
    def amp_train_step():
        with tf.GradientTape() as tape:
            predictions = model(x, training=True)
            predictions = tf.cast(predictions, tf.float32)
            loss = loss_fn(y, predictions)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss

    r_amp = benchmark_fn_gpu(amp_train_step, num_runs=50, warmup_runs=10,
                            label=f"  AMP Training (batch={batch_size})")
    throughput_amp = batch_size / (r_amp['mean_ms'] / 1000)
    r_amp['throughput_imgs_per_sec'] = float(throughput_amp)
    print(f"    → {throughput_amp:.1f} images/sec")

    results['benchmarks']['cnn_training']['amp_batch_32'] = {
        'training': r_amp,
    }

    del model, optimizer, x, y
    gc.collect()

    # Reset policy
    tf.keras.mixed_precision.set_global_policy('float32')

except Exception as e:
    print(f"  AMP failed: {e}")
    tf.keras.mixed_precision.set_global_policy('float32')

print("\n✅ CNN training benchmark complete!")

BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)

Model: SimpleResNet-18-like
Total parameters: 11,699,112

--- Batch Size: 16 ---
    Inference: 16.165 ± 0.767 ms (min=15.526, median=16.044)
    → 989.8 images/sec
    Training step: 85.005 ± 1.210 ms (min=83.688, median=84.667)
    → 188.2 images/sec

--- Batch Size: 32 ---
    Inference: 25.727 ± 1.557 ms (min=24.600, median=25.240)
    → 1243.8 images/sec
    Training step: 156.682 ± 3.381 ms (min=154.469, median=155.455)
    → 204.2 images/sec

--- Batch Size: 64 ---
    Inference: 45.170 ± 0.832 ms (min=44.267, median=44.975)
    → 1416.9 images/sec
    Training step: 299.537 ± 1.245 ms (min=298.186, median=299.372)
    → 213.7 images/sec

--- Mixed Precision Training ---
    AMP Training (batch=32): 123.585 ± 1.976 ms (min=122.594, median=123.126)
    → 258.9 images/sec

✅ CNN training benchmark complete!


In [7]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 7: Transformer Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 6: TRANSFORMER MODEL TRAINING")
print("=" * 70)

results['benchmarks']['transformer_training'] = {}

def make_transformer_model(vocab_size=30000, d_model=512, nhead=8,
                           num_layers=6, dim_ff=2048, max_seq_len=512, num_classes=2):
    inputs = tf.keras.Input(shape=(max_seq_len,), dtype=tf.int32)

    # Embedding
    x = tf.keras.layers.Embedding(vocab_size, d_model)(inputs)
    x = x * tf.math.sqrt(tf.cast(d_model, tf.float32))

    # Positional encoding (learned)
    positions = tf.keras.layers.Embedding(max_seq_len, d_model)(
        tf.range(max_seq_len)
    )
    x = x + positions

    # Transformer encoder layers
    for _ in range(num_layers):
        # Multi-head attention
        attn_output = tf.keras.layers.MultiHeadAttention(
            num_heads=nhead, key_dim=d_model // nhead
        )(x, x)
        attn_output = tf.keras.layers.Dropout(0.1)(attn_output)
        x = tf.keras.layers.LayerNormalization()(x + attn_output)

        # FFN
        ffn = tf.keras.layers.Dense(dim_ff, activation='relu')(x)
        ffn = tf.keras.layers.Dense(d_model)(ffn)
        ffn = tf.keras.layers.Dropout(0.1)(ffn)
        x = tf.keras.layers.LayerNormalization()(x + ffn)

    # Classification
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = tf.keras.layers.Dense(num_classes)(x)

    return tf.keras.Model(inputs, outputs)

transformer_configs = [
    (256, 4, 4, 1024, 128, 64, "Small-Transformer"),
    (512, 8, 6, 2048, 128, 32, "BERT-base-like"),
    (512, 8, 6, 2048, 256, 16, "BERT-long-seq"),
    (768, 12, 6, 3072, 128, 16, "BERT-large-like"),
]

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for d_model, nhead, num_layers, dim_ff, seq_len, batch_size, label in transformer_configs:
    print(f"\n--- {label} ---")
    print(f"  Config: d_model={d_model}, heads={nhead}, layers={num_layers}, "
          f"ff={dim_ff}, seq={seq_len}, batch={batch_size}")

    try:
        warmup_gpu()

        model = make_transformer_model(
            d_model=d_model, nhead=nhead, num_layers=num_layers,
            dim_ff=dim_ff, max_seq_len=seq_len
        )

        param_count = model.count_params()
        print(f"  Parameters: {param_count:,}")

        optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

        x = tf.random.uniform([batch_size, seq_len], 0, 30000, dtype=tf.int32)
        y = tf.random.uniform([batch_size], 0, 2, dtype=tf.int32)

        # Inference
        @tf.function
        def infer_fn():
            return model(x, training=False)

        r_infer = benchmark_fn_gpu(infer_fn, num_runs=30, warmup_runs=10,
                                  label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_samples_per_sec'] = float(throughput_infer)
        tokens_per_sec = (batch_size * seq_len) / (r_infer['mean_ms'] / 1000)
        r_infer['tokens_per_sec'] = float(tokens_per_sec)
        print(f"    → {throughput_infer:.1f} samples/sec, {tokens_per_sec:.0f} tokens/sec")

        # Training
        @tf.function
        def train_fn():
            with tf.GradientTape() as tape:
                predictions = model(x, training=True)
                loss = loss_fn(y, predictions)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            return loss

        r_train = benchmark_fn_gpu(train_fn, num_runs=30, warmup_runs=10,
                                  label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_samples_per_sec'] = float(throughput_train)
        tokens_per_sec_train = (batch_size * seq_len) / (r_train['mean_ms'] / 1000)
        r_train['tokens_per_sec'] = float(tokens_per_sec_train)
        print(f"    → {throughput_train:.1f} samples/sec, {tokens_per_sec_train:.0f} tokens/sec")

        results['benchmarks']['transformer_training'][label] = {
            'inference': r_infer,
            'training': r_train,
            'param_count': param_count,
            'config': {
                'd_model': d_model, 'nhead': nhead, 'layers': num_layers,
                'ff': dim_ff, 'seq_len': seq_len, 'batch': batch_size
            }
        }

        del model, optimizer, x, y
        gc.collect()

    except Exception as e:
        print(f"  SKIPPED: {e}")
        import traceback; traceback.print_exc()
        results['benchmarks']['transformer_training'][label] = {'error': str(e)}

print("\n✅ Transformer training benchmark complete!")

BENCHMARK 6: TRANSFORMER MODEL TRAINING

--- Small-Transformer ---
  Config: d_model=256, heads=4, layers=4, ff=1024, seq=128, batch=64
  Parameters: 10,839,554
    Inference: 32.040 ± 0.993 ms (min=31.617, median=31.841)
    → 1997.5 samples/sec, 255682 tokens/sec
    Training step: 104.350 ± 0.551 ms (min=103.390, median=104.276)
    → 613.3 samples/sec, 78505 tokens/sec

--- BERT-base-like ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=128, batch=32
  Parameters: 34,275,330
    Inference: 75.063 ± 0.162 ms (min=74.824, median=75.030)
    → 426.3 samples/sec, 54568 tokens/sec
    Training step: 222.502 ± 2.226 ms (min=220.973, median=221.976)
    → 143.8 samples/sec, 18409 tokens/sec

--- BERT-long-seq ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=256, batch=16
  Parameters: 34,275,330
    Inference: 56.450 ± 0.132 ms (min=56.203, median=56.425)
    → 283.4 samples/sec, 72560 tokens/sec
    Training step: 195.396 ± 1.136 ms (min=194.478, median=195.074)
    →

In [8]:
# ============================================================
# TENSORFLOW BENCHMARK - Cell 8: Memory, Transfer & Save
# ============================================================
print("=" * 70)
print("BENCHMARK 7: MEMORY & DATA TRANSFER")
print("=" * 70)

results['benchmarks']['memory'] = {}

print("\n--- Memory Transfer (CPU ↔ GPU) ---")
transfer_sizes_mb = [1, 10, 50, 100, 500, 1000]

results['benchmarks']['memory']['h2d'] = {}
results['benchmarks']['memory']['d2h'] = {}

for size_mb in transfer_sizes_mb:
    numel = size_mb * 1024 * 1024 // 4

    try:
        # Host to Device
        cpu_tensor = np.random.randn(numel).astype(np.float32)

        def h2d_fn():
            gpu_tensor = tf.constant(cpu_tensor)
            return gpu_tensor

        r_h2d = benchmark_fn_gpu(h2d_fn, num_runs=50, warmup_runs=10,
                                label=f"  H2D {size_mb} MB")
        bandwidth_h2d = size_mb / (r_h2d['mean_ms'] / 1000) / 1000
        r_h2d['bandwidth_gbs'] = float(bandwidth_h2d)
        print(f"    → {bandwidth_h2d:.2f} GB/s")
        results['benchmarks']['memory']['h2d'][f'{size_mb}MB'] = r_h2d

        # Device to Host
        gpu_tensor = tf.random.normal([numel])

        def d2h_fn():
            return gpu_tensor.numpy()

        r_d2h = benchmark_fn_gpu(d2h_fn, num_runs=50, warmup_runs=10,
                                label=f"  D2H {size_mb} MB")
        bandwidth_d2h = size_mb / (r_d2h['mean_ms'] / 1000) / 1000
        r_d2h['bandwidth_gbs'] = float(bandwidth_d2h)
        print(f"    → {bandwidth_d2h:.2f} GB/s")
        results['benchmarks']['memory']['d2h'][f'{size_mb}MB'] = r_d2h

        del cpu_tensor, gpu_tensor
        gc.collect()

    except Exception as e:
        print(f"  {size_mb} MB: ERROR - {e}")

# Allocation benchmark
print("\n--- GPU Memory Allocation ---")
results['benchmarks']['memory']['allocation'] = {}
for size_mb in [10, 100, 500, 1000]:
    numel = size_mb * 1024 * 1024 // 4
    try:
        def alloc_fn():
            t = tf.zeros([numel], dtype=tf.float32)
            return t

        r = benchmark_fn_gpu(alloc_fn, num_runs=100, warmup_runs=20,
                            label=f"  Alloc {size_mb} MB")
        results['benchmarks']['memory']['allocation'][f'{size_mb}MB'] = r
    except:
        pass

# --- Save Results ---
print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

results['timestamp'] = datetime.datetime.now().isoformat()

output_file = 'benchmark_tensorflow_results.json'
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")

# Print summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

if 'matmul' in results['benchmarks']:
    if 'float32' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float32']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP32: {mm['4096']['tflops']:.2f} TFLOPS")
    if 'float16' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float16']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP16: {mm['4096']['tflops']:.2f} TFLOPS")

if 'cnn_training' in results['benchmarks']:
    if 'batch_32' in results['benchmarks']['cnn_training']:
        ct = results['benchmarks']['cnn_training']['batch_32']
        if 'training' in ct:
            print(f"ResNet Training (batch=32): {ct['training'].get('throughput_imgs_per_sec', 'N/A'):.1f} img/s")

if 'transformer_training' in results['benchmarks']:
    if 'BERT-base-like' in results['benchmarks']['transformer_training']:
        tt = results['benchmarks']['transformer_training']['BERT-base-like']
        if 'training' in tt:
            print(f"BERT Training: {tt['training'].get('tokens_per_sec', 'N/A'):.0f} tokens/sec")

print("\n🏁 TensorFlow benchmark complete!")

BENCHMARK 7: MEMORY & DATA TRANSFER

--- Memory Transfer (CPU ↔ GPU) ---
    H2D 1 MB: 0.871 ± 0.345 ms (min=0.532, median=0.763)
    → 1.15 GB/s
    D2H 1 MB: 0.036 ± 0.017 ms (min=0.024, median=0.029)
    → 27.48 GB/s
    H2D 10 MB: 5.501 ± 0.563 ms (min=4.406, median=5.522)
    → 1.82 GB/s
    D2H 10 MB: 1.246 ± 0.442 ms (min=0.824, median=1.109)
    → 8.03 GB/s
    H2D 50 MB: 88.034 ± 7.154 ms (min=80.543, median=86.121)
    → 0.57 GB/s
    D2H 50 MB: 8.543 ± 1.165 ms (min=6.974, median=8.396)
    → 5.85 GB/s
    H2D 100 MB: 163.230 ± 5.255 ms (min=152.803, median=162.625)
    → 0.61 GB/s
    D2H 100 MB: 14.124 ± 1.046 ms (min=12.623, median=14.063)
    → 7.08 GB/s
    H2D 500 MB: 775.827 ± 15.648 ms (min=750.431, median=772.410)
    → 0.64 GB/s
    D2H 500 MB: 62.481 ± 3.179 ms (min=56.251, median=61.908)
    → 8.00 GB/s
    H2D 1000 MB: 1544.556 ± 34.713 ms (min=1492.013, median=1538.177)
    → 0.65 GB/s


2026-02-17 12:16:17.385664: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1000.00MiB (rounded to 1048576000)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-02-17 12:16:17.385900: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
2026-02-17 12:16:17.385908: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 346, Chunks in use: 324. 86.5KiB allocated for chunks. 81.0KiB in use in bin. 32.0KiB client-requested in use in bin.
2026-02-17 12:16:17.385912: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 125, Chunks in use: 120. 65.5KiB allocated for chunks. 62.2KiB in use in bin. 60.0KiB client-requested in use in bin.


  1000 MB: ERROR - {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 

--- GPU Memory Allocation ---
    Alloc 10 MB: 2.246 ± 0.528 ms (min=1.623, median=2.106)
    Alloc 100 MB: 59.206 ± 1.806 ms (min=55.972, median=58.774)
    Alloc 500 MB: 291.413 ± 6.873 ms (min=279.016, median=290.605)
    Alloc 1000 MB: 550.786 ± 6.575 ms (min=535.232, median=550.168)

SAVING RESULTS

✅ Results saved to: benchmark_tensorflow_results.json

SUMMARY
MatMul 4096x4096 FP32: 2.24 TFLOPS
MatMul 4096x4096 FP16: 4.54 TFLOPS
ResNet Training (batch=32): 204.2 img/s
BERT Training: 18409 tokens/sec

🏁 TensorFlow benchmark complete!
